## hackle_properties 세션 속성 테이블 전처리 확인

In [1]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"
TABLE_NAME = "hackle_properties"

client = bigquery.Client(project=PROJECT_ID)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [2]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.{TABLE_NAME}`
"""

df = client.query(sql).to_dataframe()
df.head()

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,session_id,user_id,language,osname,osversion,versionname,device_id
0,225279,aa2f0ec2-6e64-4937-88c0-139dc1e6ba3a,,ko,Android,11,1.0.0,aa2f0ec2-6e64-4937-88c0-139dc1e6ba3a
1,150431,f54427dc-87fc-4a71-9f22-351ebc3d83de,,ko,Android,11,1.2.1,f54427dc-87fc-4a71-9f22-351ebc3d83de
2,212190,e24fae5a-0896-4149-ac68-775ecd89c90e,,ko,Android,11,1.2.1,e24fae5a-0896-4149-ac68-775ecd89c90e
3,347659,752SqR0xcyWltcAkQzGs8ZNhvju1,,ko,Android,10,1.2.10,7b1d355a-1b31-42a8-bddc-d9f54a81e49b
4,360622,0c155764-4c9d-4e66-92c4-e6c1080c83f9,,ko,Android,10,1.2.10,0c155764-4c9d-4e66-92c4-e6c1080c83f9


## 결측치 및 데이터 정보 확인

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525350 entries, 0 to 525349
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           525350 non-null  Int64
 1   session_id   525350 non-null  str  
 2   user_id      525350 non-null  str  
 3   language     525350 non-null  str  
 4   osname       525350 non-null  str  
 5   osversion    525350 non-null  str  
 6   versionname  525350 non-null  str  
 7   device_id    525350 non-null  str  
dtypes: Int64(1), str(7)
memory usage: 79.5 MB


In [4]:
df.isna().sum()

id             0
session_id     0
user_id        0
language       0
osname         0
osversion      0
versionname    0
device_id      0
dtype: int64

## 중복값 확인

In [5]:
print("전체 행 중복:", df.duplicated().sum())
print("id 중복:", df["id"].duplicated().sum())

전체 행 중복: 0
id 중복: 0


## 범주형 컬럼 확인

In [6]:
for col in ["language", "osname", "osversion", "versionname"]:
    print(f"[{col}] 고유값 {df[col].nunique(dropna=False)}개")
    print(df[col].value_counts(dropna=False).head(10), "\n")

[language] 고유값 151개
language
ko-KR    340900
ko       164682
en-KR      9658
ko-US      4931
en         1122
en-GB       513
en-US       502
ko-JP       465
ko-CA       330
ja-KR       306
Name: count, dtype: int64 

[osname] 고유값 2개
osname
iOS        359479
Android    165871
Name: count, dtype: int64 

[osversion] 고유값 74개
osversion
16.5.1    218699
13        112112
12         32611
16.6       31126
16.3.1     22012
16.0       19622
16.2       14389
10         10229
16.1.1      9890
11          6521
Name: count, dtype: int64 

[versionname] 고유값 16개
versionname
2.0.5     309644
2.0.3     157957
2.0.0      39630
1.2.16      7847
1.2.15      7495
1.2.19      2386
1.2.10       293
1.2.8         62
2.0.4         25
1.2.4          3
Name: count, dtype: int64 



## ID 컬럼 확인

In [7]:
id_df = df[["user_id", "session_id", "device_id"]].astype("string")
id_df = id_df.apply(lambda col: col.str.strip())
blank_user = id_df["user_id"].fillna("").eq("")

print("전체 행:", len(df))
print("빈 user_id 행:", blank_user.sum())
print("user_id = session_id:", id_df["user_id"].eq(id_df["session_id"]).sum())
print("user_id = device_id:", id_df["user_id"].eq(id_df["device_id"]).sum())
print("session_id = device_id:", id_df["session_id"].eq(id_df["device_id"]).sum())

전체 행: 525350
빈 user_id 행: 82255
user_id = session_id: 92565
user_id = device_id: 0
session_id = device_id: 116567


## 충돌 분석을 위한 데이터 준비

In [8]:
# 빈 user_id는 연결 관계를 확인할 수 없으므로 점검 대상에서 제외
valid_id_df = id_df.loc[~blank_user].copy()

print("충돌 분석 대상 행:", len(valid_id_df))

충돌 분석 대상 행: 443095


## 하나의 세션에 여러 user_id가 연결됐는지 확인

In [9]:
session_df = valid_id_df[valid_id_df["session_id"].fillna("").ne("")]

session_mapping = session_df.groupby("session_id").agg(
    linked_user_count=("user_id", "nunique"),
    row_count=("user_id", "size"),
)

conflicting_sessions = session_mapping[session_mapping["linked_user_count"] > 1]

print("여러 user_id가 연결된 session_id:", len(conflicting_sessions))
display(conflicting_sessions.head(10))

여러 user_id가 연결된 session_id: 87793


,linked_user_count,row_count
session_id,,
0004F43C-3A7A-4DE4-A02B-55AFDF07E9AD,2,3
00057831-A672-4163-9C02-AB920A371F2C,2,2
0011244f-e78e-44b2-8736-d661426deba0,2,2
001384f2-7407-479c-a260-c5b525549274,2,4
0014f001-4757-48cf-a626-6bf17e910d26,2,3
00152d1f-f7f4-4607-948e-0a32b92ea32d,2,2
0018561B-F983-4E07-9B34-B73DA534B668,2,2
001B9F54-531A-4460-9440-526CC96F2463,2,2
001zjRPEGWWVou4CbhLU7JQ9K9W2,2,3


### session_id 충돌 사례 확인

In [10]:
session_conflict_sample = (
    id_df[id_df["session_id"].isin(conflicting_sessions.index)]
    [["session_id", "user_id"]]
    .drop_duplicates()
    .sort_values(["session_id", "user_id"])
    .head(20)
)

display(session_conflict_sample)

,session_id,user_id
400304,0004F43C-3A7A-4DE4-A02B-55AFDF07E9AD,1189864
7358,0004F43C-3A7A-4DE4-A02B-55AFDF07E9AD,BZulJkzkd5O2AfSkIzDXzJgPxbu2
330962,00057831-A672-4163-9C02-AB920A371F2C,1548609
331825,00057831-A672-4163-9C02-AB920A371F2C,c5gLjsxgDkRXXlQlYQPH0CkosKf2
256159,0011244f-e78e-44b2-8736-d661426deba0,1311153
279251,0011244f-e78e-44b2-8736-d661426deba0,TZlwJIxWdRRw78nfzFGwJdJIFT33
25581,001384f2-7407-479c-a260-c5b525549274,1054574
31907,001384f2-7407-479c-a260-c5b525549274,rf6vkRnsGVVY4GNei2zi0A6p7WL2
67786,0014f001-4757-48cf-a626-6bf17e910d26,1115814
8952,0014f001-4757-48cf-a626-6bf17e910d26,cKm6Q5sIrmTwlkx1qk9mnWytly52


## 하나의 기기에 여러 user_id가 연결됐는지 확인

In [11]:
device_df = valid_id_df[valid_id_df["device_id"].fillna("").ne("")]

device_mapping = device_df.groupby("device_id").agg(
    linked_user_count=("user_id", "nunique"),
    row_count=("user_id", "size"),
)

conflicting_devices = device_mapping[device_mapping["linked_user_count"] > 1]

print("여러 user_id가 연결된 device_id:", len(conflicting_devices))
display(conflicting_devices.head(10))

여러 user_id가 연결된 device_id: 87962


,linked_user_count,row_count
device_id,,
00002245-458F-4CDD-8533-B448CD43DBD2,2,2
00012620-313A-4502-9F8D-8DAB7443215B,2,2
0004241A-2F35-4A01-841E-CDB19BC5D66C,2,2
0004F43C-3A7A-4DE4-A02B-55AFDF07E9AD,2,3
00057831-A672-4163-9C02-AB920A371F2C,2,2
0007b8a1-e0f1-4e61-8cbb-271113cd5b7f,2,4
0008BE60-4C99-4E7C-BCEC-A9F1B00B0D12,2,2
0009B464-970A-4030-9990-C28F90D4D0D8,2,4
0009DD11-E7C4-4B65-AA8C-2FBECD327D23,2,2


### device_id 충돌 사례 확인

In [12]:
device_conflict_sample = (
    id_df[id_df["device_id"].isin(conflicting_devices.index)]
    [["device_id", "user_id"]]
    .drop_duplicates()
    .sort_values(["device_id", "user_id"])
    .head(20)
)

display(device_conflict_sample)

,device_id,user_id
399139,00002245-458F-4CDD-8533-B448CD43DBD2,1012728
408926,00002245-458F-4CDD-8533-B448CD43DBD2,qauR9ppJOLPISdpuqayOUMeRYiA2
383806,00012620-313A-4502-9F8D-8DAB7443215B,1319972
478153,00012620-313A-4502-9F8D-8DAB7443215B,nhdt9TAW6GXJ9UZ6DKbSZOb8lMo2
210939,0004241A-2F35-4A01-841E-CDB19BC5D66C,1234148
207855,0004241A-2F35-4A01-841E-CDB19BC5D66C,gzoLCVoHUKdSAMAGZb6lHTKvHZa2
400304,0004F43C-3A7A-4DE4-A02B-55AFDF07E9AD,1189864
7358,0004F43C-3A7A-4DE4-A02B-55AFDF07E9AD,BZulJkzkd5O2AfSkIzDXzJgPxbu2
330962,00057831-A672-4163-9C02-AB920A371F2C,1548609
331825,00057831-A672-4163-9C02-AB920A371F2C,c5gLjsxgDkRXXlQlYQPH0CkosKf2


## 전처리 확인 결과

- `id`를 기준으로 중복 여부를 확인한다.
- `user_id`, `session_id`, `device_id`가 서로 같은 행의 수를 확인한다.
- 하나의 세션 또는 기기에 여러 `user_id`가 연결된 키의 수와 사례를 확인한다.
- 숫자형 회원 ID와 영문·숫자형 익명 식별자가 함께 기록된 구조인지 사례를 바탕으로 검토한다.
- 빈 `user_id` 처리와 데이터 수정은 팀 협의 후 진행하며, 이 노트북에서는 조회·확인만 수행한다.